<a href="https://colab.research.google.com/github/fitraadhim/244107020089_MachineLearning/blob/main/JS06ANN/prak4JS06ANN.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
!pip install annoy faiss-cpu hnswlib

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 647.5/647.5 kB 3.3 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.8/18.8 MB 17.6 MB/s eta 0:00:00
  Created wheel for annoy: filename=annoy-1.17.3-cp313-cp313-linux_x86_64.whl size=520960 sha256=af19fed0fb0f9ac3f60ae16a5c0629430132259c054433f757de80cd3dd55adc
  Stored in directory: /root/.cache/pip/wheels/fc/4e/87/f1f957e7382aa370452cff98276f51686924d5415e449800ed
  Created wheel for hnswlib: filename=hnswlib-0.8.0-cp313-cp313-linux_x86_64.whl size=2679067 sha256=c7db1c46cd0816e964e12b0c893ccb28bc5f2bb304a26bedbe28ec40b5e87566
  Stored in directory: /root/.cache/pip/wheels/35/04/88/b31765a4b9957705e18065db4657e61fc8da54f50e3ef0b67e
Successfully built annoy hnswlib


In [3]:
import numpy as np
import time
from annoy import AnnoyIndex
import faiss
import hnswlib

# ===============================
# 1. Buat dataset 1 juta data 5D
# ===============================
n_data = 1_000_000   # bisa coba 100_000 dulu jika RAM terbatas
dim = 5
X = np.random.random((n_data, dim)).astype(np.float32)

# Query point
query = np.random.random((1, dim)).astype(np.float32)
k = 10

# ===============================
# 2. Annoy
# ===============================
print("=== Annoy ===")
ann_index = AnnoyIndex(dim, 'euclidean')

start = time.time()
for i in range(n_data):
    ann_index.add_item(i, X[i])
ann_index.build(10)  # 10 trees
build_time = time.time() - start

start = time.time()
neighbors = ann_index.get_nns_by_vector(query[0], k, include_distances=True)
query_time = time.time() - start

print("Build time:", build_time, "detik")
print("Query time:", query_time, "detik")
print("Neighbors:", neighbors[0][:5], "...")

# ===============================
# 3. FAISS (Flat Index)
# ===============================
print("\n=== FAISS (IndexFlatL2) ===")
faiss_index = faiss.IndexFlatL2(dim)

start = time.time()
faiss_index.add(X)
build_time = time.time() - start

start = time.time()
distances, indices = faiss_index.search(query, k)
query_time = time.time() - start

print("Build time:", build_time, "detik")
print("Query time:", query_time, "detik")
print("Neighbors:", indices[0][:5], "...")

# ===============================
# 4. HNSW (hnswlib)
# ===============================
print("\n=== HNSW (hnswlib) ===")
hnsw_index = hnswlib.Index(space='l2', dim=dim)

start = time.time()
hnsw_index.init_index(max_elements=n_data, ef_construction=200, M=16)
hnsw_index.add_items(X)
build_time = time.time() - start

hnsw_index.set_ef(50)

start = time.time()
labels, distances = hnsw_index.knn_query(query, k=k)
query_time = time.time() - start

print("Build time:", build_time, "detik")
print("Query time:", query_time, "detik")
print("Neighbors:", labels[0][:5], "...")


=== Annoy ===
Build time: 12.704422950744629 detik
Query time: 0.0002701282501220703 detik
Neighbors: [676440, 524112, 539537, 972497, 209836] ...

=== FAISS (IndexFlatL2) ===
Build time: 0.009334564208984375 detik
Query time: 0.006932258605957031 detik
Neighbors: [676440 524112 539537 972497 209836] ...

=== HNSW (hnswlib) ===
Build time: 166.48975372314453 detik
Query time: 0.00028443336486816406 detik
Neighbors: [676440 524112 539537 972497 209836] ...


In [4]:
# =====================================================
# PERCOBAAN TAMBAHAN: Menggunakan Metric Distance 'Cosine'
# =====================================================

print("\n--- PERCOBAAN DENGAN METRIC COSINE / ANGULAR ---\n")

# 1. Annoy dengan Cosine (angular)
ann_index_cos = AnnoyIndex(dim, 'angular')
start = time.time()
for i in range(n_data):
    ann_index_cos.add_item(i, X[i])
ann_index_cos.build(10)
build_time = time.time() - start

start = time.time()
neighbors_cos = ann_index_cos.get_nns_by_vector(query[0], k, include_distances=True)
query_time = time.time() - start
print("Annoy (Cosine) - Build:", build_time, "s | Query:", query_time, "s")


# 2. FAISS dengan Inner Product (Cosine similarity untuk vektor normalisasi)
faiss.normalize_L2(X)
faiss.normalize_L2(query)

faiss_index_ip = faiss.IndexFlatIP(dim)
start = time.time()
faiss_index_ip.add(X)
build_time = time.time() - start

start = time.time()
distances_ip, indices_ip = faiss_index_ip.search(query, k)
query_time = time.time() - start
print("FAISS (Inner Product) - Build:", build_time, "s | Query:", query_time, "s")


# 3. HNSW dengan Cosine
hnsw_index_cos = hnswlib.Index(space='cosine', dim=dim)
start = time.time()
hnsw_index_cos.init_index(max_elements=n_data, ef_construction=200, M=16)
hnsw_index_cos.add_items(X)
build_time = time.time() - start

hnsw_index_cos.set_ef(50)
start = time.time()
labels_cos, distances_cos = hnsw_index_cos.knn_query(query, k=k)
query_time = time.time() - start
print("HNSW (Cosine) - Build:", build_time, "s | Query:", query_time, "s")


--- PERCOBAAN DENGAN METRIC COSINE / ANGULAR ---

Annoy (Cosine) - Build: 16.48742175102234 s | Query: 0.00031566619873046875 s
FAISS (Inner Product) - Build: 0.009125709533691406 s | Query: 0.003183603286743164 s
HNSW (Cosine) - Build: 152.03882384300232 s | Query: 0.00018668174743652344 s
